## Setup — run this first

Mounts Drive, points the notebook at your project folder, and installs
what's missing. No git, no tokens.

**Your Drive folder must look like this:**

```
MyDrive/Ghana_Dropout_Project_R02/
├── config.py          <- these three at the TOP level,
├── losses.py             not inside notebooks/
├── pipeline.py
├── requirements.txt
├── notebooks/         <- the 11 notebooks
└── data-raw/
    └── ghana_dropout_study_M.xlsx
```

`results/`, `figures/`, `models/` and `data-processed/` are created for you.

Drive saves as it goes, so there is nothing to push — but see the checklist
in the last cell before you submit.


In [1]:
# ============================================================
# SETUP — Google Drive. Run first. Safe to re-run.
# ============================================================
import os, sys, subprocess
from pathlib import Path

PROJECT = "/content/drive/MyDrive/Ghana_Dropout_Project_R02"   # <-- edit if yours differs
RAW_XLSX_NAME = "ghana_dropout_study_M.xlsx"

IN_COLAB = "google.colab" in sys.modules or os.path.exists("/content")

if IN_COLAB:
    from google.colab import drive
    if not os.path.exists("/content/drive/MyDrive"):
        drive.mount("/content/drive")

    root = Path(PROJECT)
    if not root.exists():
        raise FileNotFoundError(
            f"{PROJECT} does not exist.\n"
            "Create that folder in My Drive and put config.py, losses.py, "
            "pipeline.py, requirements.txt, the notebooks/ folder and "
            "data-raw/ inside it."
        )

    # the three modules must sit at the project root, not in notebooks/
    missing = [m for m in ("config.py", "losses.py", "pipeline.py")
               if not (root / m).exists()]
    if missing:
        stray = [m for m in missing if (root / "notebooks" / m).exists()]
        msg = f"Missing from {PROJECT}: {missing}"
        if stray:
            msg += (f"\n{stray} are in notebooks/ instead. Move them UP one "
                    "level, into the project folder itself. If they stay in "
                    "notebooks/, that folder gets treated as the project root "
                    "and results/ is written in the wrong place.")
        raise FileNotFoundError(msg)

    os.chdir(root)
    os.environ["DROPOUT_REPO"] = str(root)

    # Forget any previously loaded copy of the project modules. Python keeps
    # the first version it imported for the whole session, so an edited
    # config.py is silently ignored until the runtime restarts. This makes
    # every run use the files currently in Drive.
    for _m in ("config", "losses", "pipeline"):
        sys.modules.pop(_m, None)
    if str(root) not in sys.path:
        sys.path.insert(0, str(root))

    # ---- dependencies: only install what is actually missing ------------
    need = []
    for mod, pkg in [("lightgbm", "lightgbm"), ("shap", "shap"),
                     ("catboost", "catboost"), ("xgboost", "xgboost"),
                     ("imblearn", "imbalanced-learn"), ("openpyxl", "openpyxl")]:
        try:
            __import__(mod)
        except ImportError:
            need.append(pkg)
    if need:
        print("installing:", need)
        subprocess.run(f"pip install -q {' '.join(need)}", shell=True)
    else:
        print("all dependencies present")

    # ---- raw workbook ---------------------------------------------------
    (root / "data-raw").mkdir(exist_ok=True)
    xlsx = root / "data-raw" / RAW_XLSX_NAME
    if xlsx.exists():
        print(f"raw workbook: {xlsx.name}")
    else:
        loose = list(root.glob(RAW_XLSX_NAME)) + list(root.glob(f"**/{RAW_XLSX_NAME}"))
        if loose:
            import shutil
            shutil.copy(loose[0], xlsx)
            print(f"copied {loose[0]} -> data-raw/")
        else:
            print(f"NOT FOUND: data-raw/{RAW_XLSX_NAME}\n"
                  "Notebook 1 needs it. Notebooks 2-9 read "
                  "data-processed/cleaned_data.csv instead and are fine "
                  "without it.")

    print(f"\nPROJECT : {os.getcwd()}")
else:
    print("Not in Colab — paths resolve from the project root.")


Mounted at /content/drive
installing: ['catboost']
raw workbook: ghana_dropout_study_M.xlsx

PROJECT : /content/drive/MyDrive/Ghana_Dropout_Project_R02


# Notebook 6 — Hyperparameter Search, Fully Disclosed

## What changed from R01

This notebook is why GATE-2 failed. M13 said *"No automated hyperparameter
search was conducted"*; this notebook ran `RandomizedSearchCV` at n_iter
40 / 40 / 15, and `results/optuna_trials.csv` held 20 more trials whose
generating code was in no committed notebook.

| Change | Reason |
|---|---|
| **Every trial is logged to a committed CSV** | Q26 — undisclosed exploration found by a reviewer after publication is a correction notice |
| Budget is **equal across models**, and stated | R01 gave CatBoost 15 trials at 3 folds while the others got 40 at 5 folds |
| Search scores on **AUC-PR** | R01 searched on F1 while M15 declared AUC-PR primary, so the selected configuration optimised the wrong thing |
| Search runs on the **training pool only** | GATE-1(ii) |
| The orphan artefacts are resolved explicitly | `optuna_trials.csv`, `best_parameters.json`, `best_threshold.txt` — each either regenerated by committed code or declared unused |
| The searched configuration is **not** used for the headline comparison | The research question fixes the configuration by design. The search is reported as exploratory context, not as the reported arm |

**The disclosure this notebook produces is the deliverable**, more than the
tuned model. A search that is reported is normal science.

In [2]:
# ---- bootstrap: repo-relative imports, no drive.mount, no hard-coded path ----
import sys, os
from pathlib import Path

def _find_repo(start=None):
    p = Path(start or Path.cwd()).resolve()
    for c in [p, *p.parents]:
        if (c / "config.py").exists():
            return c
    return p

REPO = Path(os.environ["DROPOUT_REPO"]) if os.environ.get("DROPOUT_REPO") else _find_repo()
if str(REPO) not in sys.path:
    sys.path.insert(0, str(REPO))

# In Colab, clone the repo first, then run:
#     import os; os.environ["DROPOUT_REPO"] = "/content/student-dropout-prediction-ghana"
# Raw pupil-level data is NOT in the repo (ethics); place it under data-raw/
# locally. Nothing below calls drive.mount().

import warnings; warnings.filterwarnings("ignore")
import numpy as np, pandas as pd
import matplotlib; matplotlib.use("Agg")
import matplotlib.pyplot as plt
import seaborn as sns

In [3]:
from config import *
from pipeline import (preprocess_inside_fold, frozen_split, cv_splits,
                      score_binary)

banner("NOTEBOOK 6 — HYPERPARAMETER SEARCH (DISCLOSED)")
OUT = run_dir("notebook06_tuning")
capture_environment(OUT)

df = pd.read_csv(CLEANED_CSV)
train_pool, test_holdout = frozen_split(df)

N_TRIALS = 40                  # IDENTICAL for every model
TUNE_SEED = SPLIT_SEED
TUNE_FOLDS = 5                 # identical for every model; no per-model override
print(f"budget: {N_TRIALS} trials per model, {TUNE_FOLDS}-fold CV on the "
      f"training pool, scored on {PRIMARY_METRIC}")
print("NO per-model override. R01 gave CatBoost 15 trials at 3 folds.")

NOTEBOOK 6 — HYPERPARAMETER SEARCH (DISCLOSED)
repo            : /content/drive/MyDrive/Ghana_Dropout_Project_R02
provenance      : NONE — set FREEZE_TAG in config.py before scoring the test set
school_handling : drop
FEATURE SET     : records   (PRIMARY — school records only)
primary metric  : auc_pr
budget: 40 trials per model, 5-fold CV on the training pool, scored on auc_pr
NO per-model override. R01 gave CatBoost 15 trials at 3 folds.


In [4]:
# ---- search space -------------------------------------------------------
from scipy.stats import loguniform, randint, uniform

SPACES = {
    "LightGBM": {
        "learning_rate": loguniform(0.005, 0.3),
        "num_leaves": randint(8, 128),
        "max_depth": randint(3, 20),
        "min_child_samples": randint(5, 40),
        "subsample": uniform(0.6, 0.4),
        "colsample_bytree": uniform(0.6, 0.4),
        "n_estimators": randint(100, 600),
    },
    "RandomForest": {
        "n_estimators": randint(100, 500),
        "max_depth": randint(3, 30),
        "min_samples_split": randint(2, 12),
        "min_samples_leaf": randint(1, 6),
        "max_features": ["sqrt", "log2", None],
    },
}
try:
    from catboost import CatBoostClassifier
    SPACES["CatBoost"] = {
        "depth": randint(3, 10),
        "learning_rate": loguniform(0.01, 0.3),
        "iterations": randint(100, 600),
        "l2_leaf_reg": loguniform(0.5, 10),
    }
except ImportError:
    print("catboost unavailable — record the omission in M13")


def build_estimator(name, params, seed):
    from lightgbm import LGBMClassifier
    from sklearn.ensemble import RandomForestClassifier
    if name == "LightGBM":
        return LGBMClassifier(objective="binary", random_state=seed,
                              verbosity=-1, **params)
    if name == "RandomForest":
        return RandomForestClassifier(random_state=seed,
                                      class_weight="balanced", **params)
    from catboost import CatBoostClassifier
    return CatBoostClassifier(random_state=seed, verbose=0,
                              allow_writing_files=False, **params)

In [5]:
# ---- the search, with every trial logged -------------------------------
from sklearn.model_selection import ParameterSampler
import time, json

folds = cv_splits(train_pool, TUNE_SEED, n_splits=TUNE_FOLDS, n_repeats=1)
prepared = []
for tr, vl in folds:
    prepared.append(preprocess_inside_fold(train_pool.iloc[tr],
                                           train_pool.iloc[vl]))
print(f"{len(prepared)} folds prepared in-fold\n")

trial_log = []
t0 = time.perf_counter()
for model_name, space in SPACES.items():
    sampler = ParameterSampler(space, n_iter=N_TRIALS, random_state=TUNE_SEED)
    print(f"--- {model_name}: {N_TRIALS} trials ---")
    for ti, params in enumerate(sampler, 1):
        scores = []
        t = time.perf_counter()
        for X_tr, y_tr, X_vl, y_vl, _ in prepared:
            est = build_estimator(model_name, params, TUNE_SEED)
            est.fit(X_tr, y_tr)
            p = est.predict_proba(X_vl)[:, 1]
            scores.append(score_binary(y_vl, p)[PRIMARY_METRIC])
        trial_log.append({
            "model": model_name, "trial": ti,
            "search_algorithm": "RandomizedSearch (ParameterSampler)",
            "scoring_metric": PRIMARY_METRIC,
            "cv_folds": TUNE_FOLDS, "seed": TUNE_SEED,
            "mean_score": float(np.mean(scores)),
            "sd_score": float(np.std(scores, ddof=1)),
            "seconds": time.perf_counter() - t,
            "params_json": json.dumps({k: (int(v) if isinstance(v, np.integer)
                                           else float(v) if isinstance(v, np.floating)
                                           else v) for k, v in params.items()}),
        })
        if ti % 10 == 0:
            print(f"    trial {ti}/{N_TRIALS} "
                  f"best so far {max(r['mean_score'] for r in trial_log if r['model']==model_name):.4f}")
    print(f"  done [{time.perf_counter()-t0:.0f}s]")

trials = pd.DataFrame(trial_log)
trials.to_csv(OUT / "search_trials_ALL.csv", index=False)
print(f"\n{len(trials)} trials logged -> search_trials_ALL.csv")
print("This file IS the M13 disclosure. Commit it.")

5 folds prepared in-fold

--- LightGBM: 40 trials ---
    trial 10/40 best so far 0.9866
    trial 20/40 best so far 0.9866
    trial 30/40 best so far 0.9892
    trial 40/40 best so far 0.9892
  done [41s]
--- RandomForest: 40 trials ---
    trial 10/40 best so far 0.9920
    trial 20/40 best so far 0.9934
    trial 30/40 best so far 0.9934
    trial 40/40 best so far 0.9934
  done [198s]
--- CatBoost: 40 trials ---
    trial 10/40 best so far 0.9802
    trial 20/40 best so far 0.9811
    trial 30/40 best so far 0.9815
    trial 40/40 best so far 0.9827
  done [568s]

120 trials logged -> search_trials_ALL.csv
This file IS the M13 disclosure. Commit it.


In [6]:
# ---- what M13 must say --------------------------------------------------
disclosure = (trials.groupby("model")
              .agg(trials=("trial", "count"),
                   cv_folds=("cv_folds", "first"),
                   scoring=("scoring_metric", "first"),
                   best_score=("mean_score", "max"),
                   worst_score=("mean_score", "min"),
                   median_score=("mean_score", "median"),
                   total_seconds=("seconds", "sum"))
              .reset_index().sort_values("best_score", ascending=False))
disclosure.to_csv(OUT / "search_disclosure_for_M13.csv", index=False)
print("SEARCH DISCLOSURE TABLE — paste into M13\n")
print(disclosure.round(4).to_string(index=False))

best_rows = trials.loc[trials.groupby("model")["mean_score"].idxmax()]
best = {r["model"]: json.loads(r["params_json"]) for _, r in best_rows.iterrows()}
(OUT / "best_parameters_REGENERATED.json").write_text(json.dumps(best, indent=2))
print("\nbest configuration per model:")
for m, p in best.items():
    print(f"  {m}: {p}")

print("\n" + "!"*72)
print("THESE CONFIGURATIONS ARE NOT USED FOR THE HEADLINE COMPARISON.")
print("The research question fixes the configuration by design (M13), and the")
print("headline arms both use config.SHARED_PARAMS with zero search trials.")
print("This search is reported as exploratory context. Say exactly that in M13.")
print("!"*72)

fig, axes = plt.subplots(1, len(disclosure), figsize=(4*len(disclosure), 3.4),
                         squeeze=False)
for ax, m in zip(axes[0], disclosure["model"]):
    s = trials[trials["model"] == m].sort_values("trial")
    ax.plot(s["trial"], s["mean_score"], ".", alpha=.6)
    ax.plot(s["trial"], s["mean_score"].cummax(), "-", lw=2, label="best so far")
    ax.set_title(m, fontsize=9); ax.set_xlabel("trial")
    ax.set_ylabel(PRIMARY_METRIC); ax.legend(fontsize=7)
plt.tight_layout(); plt.savefig(OUT / "figures/search_trajectories.png", dpi=200)
plt.close()

SEARCH DISCLOSURE TABLE — paste into M13

       model  trials  cv_folds scoring  best_score  worst_score  median_score  total_seconds
RandomForest      40         5  auc_pr      0.9934       0.9643        0.9889       157.0704
    LightGBM      40         5  auc_pr      0.9892       0.9725        0.9819        40.8775
    CatBoost      40         5  auc_pr      0.9827       0.9675        0.9776       369.8187

best configuration per model:
  CatBoost: {'depth': 7, 'iterations': 233, 'l2_leaf_reg': 1.963067235735849, 'learning_rate': 0.02102146663035812}
  LightGBM: {'colsample_bytree': 0.6163100566219055, 'learning_rate': 0.05619103481064744, 'max_depth': 6, 'min_child_samples': 23, 'n_estimators': 445, 'num_leaves': 74, 'subsample': 0.9238004184558861}
  RandomForest: {'max_depth': 28, 'max_features': 'sqrt', 'min_samples_leaf': 2, 'min_samples_split': 5, 'n_estimators': 191}

!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!
THESE CONFIGURATIONS ARE NOT USED F

In [7]:
# ---- resolve the orphan artefacts (Q4, Q26) ----------------------------
# Each of these exists in the repository with no generating code. Removal
# without disclosure is worse than leaving them, so each gets a verdict.
orphans = [
    {"artefact": "results/optuna_trials.csv",
     "status": "SUPERSEDED",
     "action": "Replaced by search_trials_ALL.csv from this notebook, which has "
               "committed generating code. Keep the old file with a README note "
               "recording that its generating code was lost, or delete it and "
               "say so in M13. Do not delete it silently.",
     "used_in_any_reported_result": False},
    {"artefact": "models/best_parameters.json",
     "status": "UNUSED",
     "action": "Regenerated as best_parameters_REGENERATED.json. Neither version "
               "is used for any reported arm. State this in M13.",
     "used_in_any_reported_result": False},
    {"artefact": "models/best_threshold.txt (0.19)",
     "status": "UNUSED — and it contradicts M15",
     "action": "M15 declares the threshold fixed at 0.5 with no optimisation. "
               "Disclose that 0.19 came from an exploratory run, was not used, "
               "and report its caseload consequence in the threshold table "
               "(Notebook 8) so a reviewer can see what it would have meant.",
     "used_in_any_reported_result": False},
    {"artefact": "results/engineered_lightgbm_results.csv (AUC-PR 0.9757)",
     "status": "SUPERSEDED, undocumented change",
     "action": "Records an earlier engineered result against the submitted "
               "0.9789 with nothing naming what changed between them. If the "
               "improvement has no named defect behind it, the instrument's "
               "rule applies: revert and report the original, or name the fix.",
     "used_in_any_reported_result": False},
]
orphan_df = pd.DataFrame(orphans)
orphan_df.to_csv(OUT / "orphan_artefact_resolution.csv", index=False)
for o in orphans:
    print(f"\n[{o['status']}] {o['artefact']}\n  -> {o['action']}")

print("\n\nCOMBINATION COUNT FLOOR for M13 (before folds):")
counted = {
    "baseline classifiers (Notebook 4)": 6,
    "imbalance strategies (Notebook 5)": 5,
    "augmentation configurations (Notebook 5b)": 5,
    "search trials, this notebook": int(len(trials)),
    "legacy Optuna trials (code lost)": 20,
    "ablation grid arms (Notebook 6b)": 9,
    "gamma x alpha cells (Notebook 6b)": 12,
}
for k, v in counted.items():
    print(f"  {k:44s} {v:>4d}")
print(f"  {'TOTAL FLOOR':44s} {sum(counted.values()):>4d}")
pd.DataFrame([counted]).T.rename(columns={0: "count"}).to_csv(
    OUT / "combination_count_floor.csv")

write_manifest(OUT, {"notebook": "06_tuning", "test_set_scored": False,
                     "trials_per_model": N_TRIALS, "cv_folds": TUNE_FOLDS,
                     "scoring_metric": PRIMARY_METRIC,
                     "total_trials": int(len(trials)),
                     "used_for_headline": False,
                     "combination_count_floor": int(sum(counted.values()))})


[SUPERSEDED] results/optuna_trials.csv
  -> Replaced by search_trials_ALL.csv from this notebook, which has committed generating code. Keep the old file with a README note recording that its generating code was lost, or delete it and say so in M13. Do not delete it silently.

[UNUSED] models/best_parameters.json
  -> Regenerated as best_parameters_REGENERATED.json. Neither version is used for any reported arm. State this in M13.

[UNUSED — and it contradicts M15] models/best_threshold.txt (0.19)
  -> M15 declares the threshold fixed at 0.5 with no optimisation. Disclose that 0.19 came from an exploratory run, was not used, and report its caseload consequence in the threshold table (Notebook 8) so a reviewer can see what it would have meant.

[SUPERSEDED, undocumented change] results/engineered_lightgbm_results.csv (AUC-PR 0.9757)
  -> Records an earlier engineered result against the submitted 0.9789 with nothing naming what changed between them. If the improvement has no named defect 

{'run_dir': 'results/notebook06_tuning/20260921T220412Z_records',
 'generated_utc': '2026-09-21T22:13:44Z',
 'git': {'commit': '',
  'branch': '',
  'dirty': False,
  'dirty_paths': [],
  'no_git': True,
  'freeze_tag': ''},
 'host': 'b2c37cded9fc',
 'platform': 'Linux-6.6.122+-x86_64-with-glibc2.39',
 'python': '3.13.15',
 'protocol': {'split_seed': 42,
  'test_size': 0.2,
  'seeds': [42, 123, 456, 789, 1024, 2048, 3333, 5555, 7777, 9999],
  'n_splits': 5,
  'n_repeats': 5,
  'primary_metric': 'auc_pr',
  'shared_params': {'n_estimators': 300,
   'num_leaves': 31,
   'learning_rate': 0.05,
   'subsample': 0.8,
   'colsample_bytree': 0.8,
   'verbosity': -1},
  'gamma_reported': 2.0,
  'alpha_reported': 0.75,
  'school_handling': 'drop',
  'feature_set': 'records',
  'active_composites': ['attendance_risk_index'],
  'fairness_threshold': 0.1,
  'drop_derived_duplicates': True,
  'suspect_column_actions': {'social_studies_exam_score': 'keep'}},
 'notebook': '06_tuning',
 'test_set_score

---

## Before you submit

Drive has already saved everything — nothing to push. But two things still
have to happen before submission, and neither is automatic.


In [8]:
# ---- what this run produced, and what is still owed ----
import os, sys
from pathlib import Path

try:
    latest = sorted(Path(OUT).parent.glob("*"))[-1]
    files = sorted(p.relative_to(OUT).as_posix() for p in Path(OUT).rglob("*")
                   if p.is_file())
    print(f"run directory : {Path(OUT).relative_to(REPO)}")
    print(f"files written : {len(files)}")
    for f in files:
        print("   ", f)
except Exception as e:
    print("no run directory recorded in this session:", e)

print("""
────────────────────────────────────────────────────────────────────
STILL OWED BEFORE SUBMISSION — neither happens by itself

1. UPLOAD THE PROJECT TO GITHUB, ONCE.
   Q4 failed because the repository was not runnable from a clone. Drive
   is fine for working; the repo is the deliverable. When the analysis is
   finished, drag the whole project folder into GitHub in one upload —
   EXCEPT data-raw/ and anything holding pupil rows. The notebooks resolve
   paths relative to the project root, so they run from a clone unchanged.

   Do NOT upload:  data-raw/, data-processed/cleaned_data.csv,
                   any *_snapshot.csv
   DO upload:      config.py, losses.py, pipeline.py, notebooks/,
                   requirements.txt, README.md, and all of results/

2. SET config.FREEZE_TAG BEFORE SCORING THE TEST SET.
   Without git there is no commit hash to anchor the freeze to. Put a
   fixed dated string in config.py — e.g. "R02-freeze-2026-09-25-1430" —
   at the moment you freeze the configuration, and never revise it.
   Notebook 8 refuses to score the test set until it is set.
────────────────────────────────────────────────────────────────────""")


run directory : results/notebook06_tuning/20260921T220412Z_records
files written : 9
    RUN_MANIFEST.json
    best_parameters_REGENERATED.json
    combination_count_floor.csv
    environment_versions.csv
    figures/search_trajectories.png
    orphan_artefact_resolution.csv
    pip_freeze.txt
    search_disclosure_for_M13.csv
    search_trials_ALL.csv

────────────────────────────────────────────────────────────────────
STILL OWED BEFORE SUBMISSION — neither happens by itself

1. UPLOAD THE PROJECT TO GITHUB, ONCE.
   Q4 failed because the repository was not runnable from a clone. Drive
   is fine for working; the repo is the deliverable. When the analysis is
   finished, drag the whole project folder into GitHub in one upload —
   EXCEPT data-raw/ and anything holding pupil rows. The notebooks resolve
   paths relative to the project root, so they run from a clone unchanged.

   Do NOT upload:  data-raw/, data-processed/cleaned_data.csv,
                   any *_snapshot.csv
   DO up